# Quant Skills — Lesson 2: Real Market Data and Volatility

**Goal:** pull real prices, measure how much a stock really moves, and turn that into
**expected moves** you can use to pick option strikes.

**How to run it:** click **Runtime → Run all**. That's it — every cell runs on its own.

**The only lines you'll ever edit** are marked with `# <- EDIT`. Everything else stays as-is.

## Step 1 — Load the tools
This installs the free price-data library. Takes about 10 seconds.

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Tools loaded.")

## Step 2 — Download two years of daily prices

In [ ]:
TICKERS = ["SPY", "NVDA"]          # <- EDIT: try ["QQQ", "AMZN"] or ["SPY", "AAPL", "GOOGL"]

raw = yf.download(TICKERS, period="2y", interval="1d", auto_adjust=True, progress=False)
close = raw["Close"]
if isinstance(close, pd.Series):
    close = close.to_frame(TICKERS[0])
close = close[TICKERS].dropna()

print(f"Downloaded {len(close)} trading days, from {close.index[0].date()} to {close.index[-1].date()}")
print()
print("Latest closing prices:")
print(close.tail(1).round(2).to_string())

## Step 3 — Daily returns

A **return** is the % change from one day's close to the next. Quants work in returns, not prices,
because a $5 move means something very different on a $50 stock and a $700 stock.

In [ ]:
returns = close.pct_change().dropna()

summary = pd.DataFrame({
    "Avg day":     returns.mean(),
    "Biggest up":  returns.max(),
    "Biggest down": returns.min(),
    "% days up":   (returns > 0).mean(),
})
print((summary * 100).round(2).astype(str) + "%")

## Step 4 — Volatility: how much it really moves

**Volatility** is the standard deviation of daily returns — the typical size of a day's move.
To turn daily into yearly, multiply by **√252 ≈ 15.87** (the square-root rule from Lesson 1).

This is called **realized volatility**: what the stock actually did.

In [ ]:
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)
recent_vol = returns.tail(20).std() * np.sqrt(252)

vol_table = pd.DataFrame({
    "Typical daily move": daily_vol,
    "Annual vol (2 yrs)": annual_vol,
    "Annual vol (last 20 days)": recent_vol,
})
print((vol_table * 100).round(1).astype(str) + "%")
print()
for t in TICKERS:
    ratio = annual_vol[t] / annual_vol[TICKERS[0]]
    if t != TICKERS[0]:
        print(f"{t} moves about {ratio:.1f}x as much as {TICKERS[0]}.")

### Volatility changes over time

The line below shows 20-day volatility rolling through the last two years. Spikes are fear;
quiet stretches are calm markets. Premium sellers make their money when vol is high and falling.

In [ ]:
rolling_vol = returns.rolling(20).std() * np.sqrt(252) * 100

plt.figure(figsize=(10, 4.5))
for t in TICKERS:
    plt.plot(rolling_vol.index, rolling_vol[t], label=t, linewidth=1.4)
plt.title("20-day realized volatility (annualized %)")
plt.ylabel("Volatility %"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 5 — Expected move: where volatility meets your strikes

This is the part that pays off directly in your trading.

**Expected move (1 standard deviation) = price × volatility × √(days ÷ 252)**

About **68%** of the time, price finishes inside that range. A strike right at the edge of it
is roughly a **16-delta** option — the strike zone we use for credit spreads.

In [ ]:
HORIZONS = {"1 week": 5, "2 weeks": 10, "1 month": 21}   # <- EDIT: trading days

for t in TICKERS:
    price = close[t].iloc[-1]
    vol = recent_vol[t]
    print(f"{t}  —  price ${price:,.2f}  ·  using recent vol {vol*100:.1f}%")
    for name, days in HORIZONS.items():
        move = price * vol * np.sqrt(days / 252)
        print(f"   {name:8s}: ±${move:,.2f}   →  range ${price-move:,.2f} to ${price+move:,.2f}")
    print()

## Step 6 — Implied vs. realized volatility (your edge in premium selling)

**Implied volatility (IV)** is what the options market *expects*. **Realized** is what actually happened.

- **IV above realized** → options are priced rich → favors **selling** premium (credit spreads)
- **IV below realized** → options are cheap → favors **buying** (debit spreads, long options)

Look up each ticker's IV in thinkorswim (it's on the Trade tab next to the option chain) and type it below.

In [ ]:
IMPLIED_VOL = {"SPY": 0.15, "NVDA": 0.45}   # <- EDIT: type today's IV from thinkorswim (0.15 = 15%)

for t in TICKERS:
    if t not in IMPLIED_VOL:
        print(f"{t}: add its IV to IMPLIED_VOL above to compare.")
        continue
    iv, rv = IMPLIED_VOL[t], recent_vol[t]
    gap = iv - rv
    verdict = ("options look RICH — favors selling premium" if gap > 0.02 else
               "options look CHEAP — favors buying" if gap < -0.02 else
               "roughly fair — no strong edge either way")
    print(f"{t}: implied {iv*100:.1f}%  vs  realized {rv*100:.1f}%  →  gap {gap*100:+.1f} pts  →  {verdict}")

## Step 7 — How bad can it get? Max drawdown

The worst peak-to-trough drop over the period. This is what you'd have sat through holding the stock.

In [ ]:
for t in TICKERS:
    peak = close[t].cummax()
    dd = (close[t] - peak) / peak
    worst = dd.min()
    when = dd.idxmin().date()
    print(f"{t}: worst drawdown {worst*100:.1f}%  (bottom on {when})")

## Your exercises

1. Change `TICKERS` in Step 2 to `["QQQ", "AMZN"]` and run all again. Which is more volatile?
2. In Step 5, find NVDA's **1-month expected move**. Would a call credit spread with its short strike
   *inside* that range be a 16-delta trade, or something riskier?
3. In Step 6, type in **real IVs** from thinkorswim for SPY and NVDA. Which one is better for selling premium right now?

**Next lesson:** options pricing — building Black-Scholes and the Greeks so you can see exactly where
delta and theta come from.